# Rust 112: Iterators in Depth — Practice

Companion exercises for `12_iterators-in-depth_e.ipynb`. Every section mirrors the section of the same number in the example notebook, and there is one exercise for each of its twenty demonstration cells, so a `**See**` reference lands on the exact code that shows the technique.

The checks at the bottom of each cell are the grader. Edit only the parts marked `TODO`; when a cell runs without panicking you will see an `Exercise N.M passed` line.


### Contents

- [0. How to Complete These Exercises](#0-how-to-complete-these-exercises)
- [1. The Trait: One Required Method](#1-the-trait-one-required-method)
- [2. A Custom Iterator from Scratch](#2-a-custom-iterator-from-scratch)
- [3. Adaptors vs Consumers](#3-adaptors-vs-consumers)
- [4. Laziness and Short-Circuiting](#4-laziness-and-short-circuiting)
- [5. `collect`: The Fan-Out](#5-collect-the-fan-out)
- [6. `for` Loops and the Three Iterators](#6-for-loops-and-the-three-iterators)
- [7. Beyond `next`: `fold`, `try_fold` and Friends](#7-beyond-next-fold-try-fold-and-friends)
- [8. Returning Iterators: `impl Iterator`](#8-returning-iterators-impl-iterator)
- [9. Rolling Your Own Adaptor](#9-rolling-your-own-adaptor)
- [10. Reading the Compiler: E-Codes](#10-reading-the-compiler-e-codes)
- [11. The Cost Model](#11-the-cost-model)
- [14. Review: Mixed Problems](#14-review-mixed-problems)

§1–§11 each have one exercise for every demonstration cell of the example notebook (the Contents lists only the exercise-bearing sections; §12's control panel and §13's cheat sheet are reading, and §14's mixed problems assemble several sections at once). When a check fails, the fastest route back is §13 of the example notebook, which is the cheat sheet.


## 0. How to Complete These Exercises

This notebook runs on the same **evcxr** kernel as the example notebook. Three consequences shape how you work:

1. **No `fn main`.** The body of a cell *is* the body of `main`. Type statements directly.
2. **State persists.** A binding from one cell is visible in every cell below it — including `use` lines.
3. **Items are order-independent.** A `struct` or `impl` may be used above the cell that defines it.

> **Item names must be unique across the whole notebook.** Every cell shares one namespace; two functions with the same name are a redefinition error even in different sections. The exercise names here are unique by construction — do not introduce colliding helpers.

### The stub convention

Each exercise cell declares a signature, a doc comment stating the contract, and a `// --- check (do not modify) ---` block of `assert_eq!`/`assert!` calls. The body you must fill in is `todo!()`, which compiles but **panics** when executed. Replace it and re-run; an `Exercise N.M passed` line means you are done.

### Three notes specific to this chapter

- **Given code is complete and stays.** Cells hand you iterators so the stub has something to drive: `Ladder` (§2.1), `Countdown` (§2.2), `Dedup` (§9.1) and `Fib` (§14.3) are declared with their constructors, and given impls stay exactly as written — the empty `impl FusedIterator for Ladder {}` is the promise, not an omission. This is the one-namespace rule working *for* you.
- **Factories return placeholder chains, not `todo!()`.** A `-> impl Iterator` stub whose body is `todo!()` cannot compile: the hidden type would be inferred as `!`, and `!` is not an `Iterator` (E0277). So §8.1 and §11.1's stubs ship a **placeholder chain** — it compiles, it yields wrong answers, and the check block's `assert` panic is the unsolved signal. Replace the placeholder with the real chain; the doc comment marks the spot.
- **Infinite sources are bounded by construction.** §4.1 and §14.3 build chains with no end; every check bounds them with `take`/`take_while` before collecting. That is laziness doing its job — if one of your bodies collects an unbounded chain, the cell does not error, it hangs, and that hang *is* the §4 lesson.
- **The E-codes are live here.** The wrong shape trips the exact diagnostics the example notebook catalogues in §10: forget the `move` on §8.1's closure and the borrow checker refuses the factory; `.rev()` on an iterator without `DoubleEndedIterator` is E0599; and §10.1's dodges are the stable tools for E0689, E0283 and E0599. When you see one, you are one step from the lesson.

### Expect warnings until you finish

Three warning families are normal and self-cleaning:

- `unused variable` — every unsolved stub whose parameters are untouched;
- `unused import` — the `use` lines that serve a stub you have not solved yet;
- `never read`/`never used` — stub-induced dead code: the check blocks that would read a given field sit below an unsolved stub.

Each warning disappears when you implement the body that needs it. If a diagnostic appears that is not in one of those families, it is yours.

**See**: §0 and §10 of the example notebook.


### Exercise 0.1 — The smoke test

**Goal**: prove the kernel compiles your items and runs your code, before any real exercise.

**Objectives**

- implement `kernel_pulse` to return the chapter's number
- keep the signature unchanged

**See**: §0.1 of the example notebook.

**Hint**: the body is one tail expression; no semicolon.


In [ ]:
/// Return the chapter number — the smallest possible proof the kernel is alive.
///
/// TODO: replace `todo!()` with the chapter number.
fn kernel_pulse() -> u32 {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(kernel_pulse(), 12);
println!("Exercise 0.1 passed");


### Exercise 1.1 — The desugar, written out

**Goal**: sum a slice by driving `next` yourself — the loop `for` performs on your behalf, with nothing hidden.

**Objectives**

- implement `sum_by_hand` using a `while let` over `it.next()` — no `for`, no `.sum()`
- keep the signature unchanged

**See**: §1.1 of the example notebook.

**Hint**: bind `let mut it = v.iter();` first, then `while let Some(&x) = it.next()` — the `&x` pattern copies the `u32` out (chapter 05's rule).


In [ ]:
/// Sum a slice by hand: the for-loop desugar, written out with next().
///
/// TODO: replace `todo!()` with a `while let Some(..) = it.next()` loop.
fn sum_by_hand(v: &[u32]) -> u32 {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(sum_by_hand(&[10, 20, 30]), 60);
assert_eq!(sum_by_hand(&[]), 0);
assert_eq!(sum_by_hand(&[7]), 7);
println!("Exercise 1.1 passed");


### Exercise 2.1 — An iterator from scratch

**Goal**: implement `next` for `Ladder`, a stride iterator — the whole protocol in four lines, plus the marker promise for free.

**Given**: the `Ladder` struct and its constructor are complete; only `next` in the `Iterator` impl is missing. The empty `FusedIterator` impl below it is already legal — it promises what your `next` will deliver (once `None`, always `None`).

**Objectives**

- implement `next`: while the cursor is below `end`, yield it and advance by `step`; otherwise `None`
- collect a `Ladder` with `collect`, and drive one with `for` — both must work with nothing beyond your impl
- keep the struct, the constructor and the marker impl unchanged

**See**: §2.1 of the example notebook.

**Hint**: the guard is `self.next < self.end`; yield the cursor first, then advance it by `step`.


In [ ]:
// (std::iter::FusedIterator is imported here for the rest of the notebook.)
use std::iter::FusedIterator;

struct Ladder {
    next: u32, // the cursor
    end: u32,
    step: u32,
}
impl Ladder {
    fn new(start: u32, end: u32, step: u32) -> Self {
        Ladder { next: start, end, step }
    }
}
impl Iterator for Ladder {
    type Item = u32;
    /// TODO: replace `todo!()` with the cursor guard, yield, and advance.
    fn next(&mut self) -> Option<u32> {
        todo!()
    }
}
// Given, 2.1's lesson applied: the promise is free once next() stays None.
impl FusedIterator for Ladder {}

// --- check (do not modify) ---
let ys: Vec<u32> = Ladder::new(0, 10, 3).collect();
assert_eq!(ys, vec![0, 3, 6, 9]);
assert_eq!(Ladder::new(5, 5, 1).next(), None); // start at the end: empty
let mut n = 0;
for _ in Ladder::new(0, 6, 2) {
    n += 1; // `for` needs nothing beyond the Iterator impl
}
assert_eq!(n, 3);
println!("Exercise 2.1 passed");


### Exercise 2.2 — The exact-length contract

**Goal**: give `Countdown` an exact `size_hint` — both bounds the truth — and read the upgrade it buys.

**Given**: the `Countdown` struct, its constructor and its `next` (which yields `from` down to 1) are complete; only `size_hint` is missing. The empty `impl ExactSizeIterator for Countdown {}` below is legal *because* your hint will be exact — do not add methods to it.

**Objectives**

- implement `size_hint` so both bounds equal the number of items remaining
- observe the contract move: `len()` and `count()` agree with your hint after every pull
- keep the signatures and the given marker impl unchanged

**See**: §2.2 of the example notebook.

**Hint**: both bounds are the same number — `remaining`, widened with `as usize`.


In [ ]:
struct Countdown {
    remaining: u32,
}
impl Countdown {
    fn new(from: u32) -> Self {
        Countdown { remaining: from }
    }
}
impl Iterator for Countdown {
    type Item = u32;
    fn next(&mut self) -> Option<u32> {
        if self.remaining == 0 {
            None
        } else {
            let out = self.remaining;
            self.remaining -= 1;
            Some(out)
        }
    }
    /// TODO: replace `todo!()` with the exact remaining count, both bounds.
    fn size_hint(&self) -> (usize, Option<usize>) {
        todo!()
    }
}
// Given: the hint you just wrote is exact, so this marker impl is legal.
impl ExactSizeIterator for Countdown {}

// --- check (do not modify) ---
let mut c = Countdown::new(3);
assert_eq!(c.size_hint(), (3, Some(3)));
assert_eq!(c.next(), Some(3));
assert_eq!(c.size_hint(), (2, Some(2))); // the cursor moved, the hint followed
assert_eq!(c.len(), 2); // ExactSizeIterator's default method reads the hint
assert_eq!(c.count(), 2); // count() drains: it IS the hint, proven
assert_eq!(Countdown::new(0).size_hint(), (0, Some(0)));
println!("Exercise 2.2 passed");


### Exercise 3.1 — The workhorse pipeline

**Goal**: chain `filter` → `map` → `sum` on borrowed items — the pipeline family where adaptors wrap and only the consumer drives.

**Info**: `filter`'s predicate receives `&&u32` (the item is already a `&u32` over the slice); the `|&x|` pattern in the example notebook's cell is the deref ladder paying off.

**Objectives**

- implement `doubled_evens` as one chain: `filter` even items, `map` to double them, `sum` the result
- keep the signature unchanged; the chain must borrow `v` — no `into_iter`
- confirm the shape: swap the chain to `map` first and nothing breaks — but a `bare .map(..);` without a consumer is `unused_must_use`, the §3 rule

**See**: §3.1 of the example notebook.

**Hint**: `v.iter().filter(|&&x| x % 2 == 0).map(|&x| x * 2).sum()` — two derefs in `filter`'s pattern, one in `map`'s.


In [ ]:
/// Sum of the even items, each doubled — one borrowed chain.
///
/// TODO: replace `todo!()` with the filter-map-sum chain.
fn doubled_evens(v: &[u32]) -> u32 {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(doubled_evens(&[1, 2, 3, 4, 5]), 12); // 2*2 + 4*2
assert_eq!(doubled_evens(&[1, 3, 5]), 0); // nothing passes the filter
assert_eq!(doubled_evens(&[]), 0);
println!("Exercise 3.1 passed");


### Exercise 3.2 — The shelf: `filter_map`, `enumerate`, `zip`

**Goal**: use the three shelf adaptors that replace half of all hand-written loops — drop failures, tag with index, pair two sources.

**Objectives**

- implement `parse_positions` to pair each *parsable* row with its original index: `enumerate` for the positions, `filter_map` with `parse().ok()` for the survival, `collect` for the target
- implement `pair_with_index` to zip the rows with their indices — one chain, the signature owns the lifetime that ties the output to the input
- keep the signatures unchanged

**See**: §3.2 of the example notebook.

**Hint**: `(i, s)` arrives in `filter_map`'s closure as `&(usize, &str)` — pattern it as `|(i, s)|`, then `s.parse().ok().map(|v| (*i, v))`; for the zip, the given signature's `'a` is exactly what `v.iter().zip(0..)` yields.


In [ ]:
/// Index and parsed value for every row that parses; rows that fail vanish.
///
/// TODO: replace `todo!()` with the enumerate + filter_map chain.
fn parse_positions(rows: &[&str]) -> Vec<(usize, u32)> {
    todo!()
}
/// Zip the source with its indices: the signature names the lifetime.
///
/// TODO: replace `todo!()` with the zip chain.
fn pair_with_index<'a>(v: &[&'a str]) -> Vec<(usize, &'a str)> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(parse_positions(&["10", "x", "30"]), vec![(0, 10), (2, 30)]);
assert_eq!(parse_positions(&["a", "b"]), vec![]);
assert_eq!(pair_with_index(&["es", "nq"]), vec![(0, "es"), (1, "nq")]);
assert_eq!(pair_with_index(&[]), vec![]);
println!("Exercise 3.2 passed");


### Exercise 4.1 — Laziness: an infinite source, safely bounded

**Goal**: build a chain over an *infinite* source and bound it with `take` — possible only because adaptors never run and the consumer stops pulling.

**Given**: `Naturals::new()` is a given infinite iterator (`next` never returns `None`).

**Objectives**

- implement `first_powers` to return the squares of the first `n` naturals — the chain must start from `Naturals::new()`, not a finite range
- implement `bounded_sum` to sum the first `n` items of the same infinite source
- keep the signatures unchanged; the `use` line for `Cell`/`Rc` from §4.1's proof is not needed here — the bound is `take`, not a counter

**See**: §4.1 of the example notebook.

**Hint**: `Naturals::new().map(|x| x * x).take(n).collect()` — the `take` is what makes infinity collectable.


In [ ]:
/// An infinite source: next() never returns None. Do not collect it unbounded.
struct Naturals {
    cursor: u64,
}
impl Naturals {
    fn new() -> Self {
        Naturals { cursor: 0 }
    }
}
impl Iterator for Naturals {
    type Item = u64;
    fn next(&mut self) -> Option<u64> {
        let out = self.cursor;
        self.cursor += 1;
        Some(out) // never None: infinite
    }
}
impl FusedIterator for Naturals {}

/// The squares of the first n naturals — from an infinite source.
///
/// TODO: replace `todo!()` with the map + take chain.
fn first_powers(n: usize) -> Vec<u64> {
    todo!()
}
/// Sum of the first n naturals — the consumer stops, the source does not.
///
/// TODO: replace `todo!()` with the take + sum chain.
fn bounded_sum(n: usize) -> u64 {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(first_powers(4), vec![0, 1, 4, 9]); // four naturals, squared
assert_eq!(first_powers(0), vec![]);
assert_eq!(bounded_sum(5), 10); // 0+1+2+3+4
assert_eq!(bounded_sum(0), 0);
println!("Exercise 4.1 passed");


### Exercise 4.2 — Short-circuit: stop at the answer

**Goal**: find the *first* violation in a long series with a short-circuiting consumer, and prove it stops by timing nothing.

**Info**: `position` returns the index of the first item passing the predicate — and stops pulling the moment it finds one.

**Objectives**

- implement `first_breach` to return the index of the first price outside `[lo, hi]`, or `None` if all hold — via `position`, not a `for` loop
- implement `all_in_band` to answer the same question as a boolean — via `all`
- keep the signatures unchanged

**See**: §4.2 of the example notebook.

**Hint**: both bodies are one chain over `iter().copied()`; `position(|p| *p < lo || *p > hi)` and `all(|p| *p >= lo && *p <= hi)` are the consumers that stop early.


In [ ]:
/// Index of the first price outside the band, or None — short-circuits.
///
/// TODO: replace `todo!()` with the position chain.
fn first_breach(px: &[f64], lo: f64, hi: f64) -> Option<usize> {
    todo!()
}
/// Do all prices hold the band? — stops at the first breach.
///
/// TODO: replace `todo!()` with the all chain.
fn all_in_band(px: &[f64], lo: f64, hi: f64) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(first_breach(&[100.0, 101.0, 90.0, 88.0], 95.0, 105.0), Some(2));
assert_eq!(first_breach(&[100.0, 101.0], 95.0, 105.0), None);
assert!(all_in_band(&[100.0, 101.0], 95.0, 105.0));
assert!(!all_in_band(&[100.0, 90.0], 95.0, 105.0));
println!("Exercise 4.2 passed");


### Exercise 4.3 — `chunks` vs `windows`: the two slice views

**Goal**: compute the same slice twice — disjoint blocks with a ragged tail, and sliding panes with the `k − n + 1` count — and know which one is the moving average.

**Objectives**

- implement `into_blocks` to cut the slice into consecutive `n`-sized blocks (the last may be shorter) as owned `Vec`s
- implement `moving_sums` to compute the sum of each `n`-wide window
- keep the signatures unchanged; the count relationship is asserted, not recomputed

**See**: §4.3 of the example notebook.

**Hint**: `v.chunks(n).map(|c| c.to_vec()).collect()` and `v.windows(n).map(|w| w.iter().sum()).collect()` — both items are already slices.


In [ ]:
/// Disjoint blocks of size n; the tail block may be shorter.
///
/// TODO: replace `todo!()` with the chunks chain.
fn into_blocks(v: &[u32], n: usize) -> Vec<Vec<u32>> {
    todo!()
}
/// Sum of each sliding n-wide window.
///
/// TODO: replace `todo!()` with the windows chain.
fn moving_sums(v: &[u32], n: usize) -> Vec<u32> {
    todo!()
}

// --- check (do not modify) ---
let data: Vec<u32> = (1..=7).collect();
assert_eq!(into_blocks(&data, 3), vec![vec![1, 2, 3], vec![4, 5, 6], vec![7]]);
assert_eq!(moving_sums(&data, 2), vec![3, 5, 7, 9, 11, 13]); // 7 - 2 + 1 panes
assert_eq!(moving_sums(&data, 3).len(), 5); // 7 - 3 + 1
assert_eq!(moving_sums(&data, 7), vec![28]); // exactly one full-width pane
println!("Exercise 4.3 passed");


### Exercise 5.1 — The collect zoo

**Goal**: land one chain in each of two `FromIterator` targets — a `String` built from chars, and a `BTreeMap` built from pairs.

**Info**: `collect`'s target never appears in the chain — it comes from the binding's type (or the turbofish). Here the signature carries it.

**Objectives**

- implement `csv_row` to join the rows into one `String` with `,` between them — via `chars` and `collect`, no `for` loop. Rows yield their own chars; `flat_map` removes the nesting (§3's shelf row)
- implement `char_counts` to fold the chars into a `BTreeMap<char, usize>` — `entry(c).or_insert(0)` grows each count. **`collect` cannot do this job**: a repeated key *overwrites* (last `(c, 1)` wins), it never adds
- keep the signatures unchanged

**See**: §5.1 of the example notebook.

**Hint**: the row-join is `flat_map(|r| r.chars())` with a `,` chained in front of every row but the first; the counting fold is `fold(BTreeMap::new(), |mut m, c| { *m.entry(c).or_insert(0) += 1; m })` — chapter 06's `entry`, on the sorted map.


In [ ]:
// (BTreeMap is imported here for the rest of the notebook.)
use std::collections::BTreeMap;

/// Join rows into one comma-separated String, char by char, no loop.
///
/// TODO: replace `todo!()` with the flat_map + collect chain.
fn csv_row(rows: &[&str]) -> String {
    todo!()
}
/// Count occurrences: entry grows the count, fold carries the map.
/// (collect of (c, 1) pairs would overwrite, not add — try it.)
///
/// TODO: replace `todo!()` with the entry-based fold.
fn char_counts(word: &str) -> BTreeMap<char, usize> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(csv_row(&["es", "nq"]), "es,nq");
assert_eq!(csv_row(&["a"]), "a");
let counts = char_counts("book");
assert_eq!(counts[&'b'], 1); // BTreeMap<char, _> indexes by &char
assert_eq!(counts[&'o'], 2); // the count GREW - collect would have left 1
println!("Exercise 5.1 passed");


### Exercise 5.2 — The fail-fast collect

**Goal**: parse every row or fail on the first bad one — `collect` into `Result<Vec<u32>, _>`, the consumer edition of `?`.

**Objectives**

- implement `parse_all` to collect the parse results into `Result<Vec<u32>, std::num::ParseIntError>` — the signature carries the target; the chain is one `map` plus `collect`
- implement `first_bad` to return the index of the first row that fails, or `None` — `map` with `parse().is_err()` first, then `position` (§4.2's consumer again)
- keep the signatures unchanged

**See**: §5.2 of the example notebook.

**Hint**: the target type is spelled once in the signature: `Result<Vec<u32>, ParseIntError>` — inside the body the chain is just `rows.iter().map(|r| r.parse::<u32>()).collect()`.


In [ ]:
/// Parse every row or fail fast: collect's Result target short-circuits.
///
/// TODO: replace `todo!()` with the map + collect chain.
fn parse_all(rows: &[&str]) -> Result<Vec<u32>, std::num::ParseIntError> {
    todo!()
}
/// Index of the first unparseable row, or None.
///
/// TODO: replace `todo!()` with the map + position chain.
fn first_bad(rows: &[&str]) -> Option<usize> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(parse_all(&["10", "20"]), Ok(vec![10, 20]));
let err = parse_all(&["10", "x", "30"]).unwrap_err();
assert!(err.to_string().contains("invalid digit")); // the ParseIntError
assert_eq!(first_bad(&["10", "x", "y"]), Some(1)); // first failure wins
assert_eq!(first_bad(&["1", "2"]), None);
println!("Exercise 5.2 passed");


### Exercise 6.1 — The three forms: who survives?

**Goal**: read a `Vec` one way and consume it another — then feel the E0382 that sorts them.

**Objectives**

- implement `borrowed_sum` to sum a slice via `iter()` — the source survives, so the check prints it afterwards
- implement `into_count` to count a `Vec` via `into_iter()` — the source is *moved*, which is why the signature takes it by value
- keep the signatures unchanged; note in the check block which call could not use the other's form

**See**: §6.1 of the example notebook.

**Hint**: `v.iter().copied().sum()` keeps the caller alive; `fn into_count(v: Vec<u32>)` is the signature shape that *promises* the move.


In [ ]:
/// Sum by borrowing: the caller's slice survives untouched.
///
/// TODO: replace `todo!()` with the iter chain.
fn borrowed_sum(v: &[u32]) -> u32 {
    todo!()
}
/// Count by consuming: the Vec is moved in, and is gone afterwards.
///
/// TODO: replace `todo!()` with the into_iter chain.
fn into_count(v: Vec<u32>) -> usize {
    todo!()
}

// --- check (do not modify) ---
let prices = vec![10, 20, 30];
assert_eq!(borrowed_sum(&prices), 60);
assert_eq!(prices.len(), 3); // survived: borrowed_sum only read it
assert_eq!(into_count(prices), 3);
// println!("{prices:?}"); // error[E0382]: borrow of moved value: `prices`
println!("Exercise 6.1 passed");


### Exercise 6.2 — In place: `iter_mut` and the zip

**Goal**: mutate a slice in place — index-aware for one stub, destination-paired for the other — without moving anything.

**Objectives**

- implement `scale_at` to multiply each price by `1.0 + i as f64 / 100.0` in place — `iter_mut().enumerate()` is the idiom
- implement `pair_rates` to fill the empty `slots` with `100.0 * (1.0 + rate)` — `slots.iter_mut().zip(rates)` is the pairing
- keep the signatures unchanged; both must mutate through `&mut [f64]` only

**See**: §6.2 of the example notebook.

**Hint**: `for (i, p) in v.iter_mut().enumerate()` gives `&mut f64` items — `*p *= ...`; `zip` ends at whichever side is shorter, which is the guard.


In [ ]:
/// Index-aware in-place scaling: price[i] *= 1.0 + i/100.
///
/// TODO: replace `todo!()` with the iter_mut + enumerate loop.
fn scale_at(v: &mut [f64]) {
    todo!()
}
/// Pair each slot with its rate: 100.0 * (1.0 + rate) written in place.
///
/// TODO: replace `todo!()` with the iter_mut + zip loop.
fn pair_rates(slots: &mut [f64], rates: &[f64]) {
    todo!()
}

// --- check (do not modify) ---
let mut px = vec![100.0, 100.0, 100.0];
scale_at(&mut px);
assert!(px[0] >= 100.0 && px[2] > px[0]); // i = 0 scales by exactly 1.0
let mut slots = vec![0.0; 3];
pair_rates(&mut slots, &[0.1, 0.2, 0.3]);
assert_eq!(slots[2], 130.0);
let mut short = vec![0.0; 2];
pair_rates(&mut short, &[0.1, 0.2, 0.3]); // zip ends at the shorter side
assert_eq!(short.len(), 2); // the third rate had no slot to land in
println!("Exercise 6.2 passed");


### Exercise 7.1 — `fold`: state that `map` cannot hold

**Goal**: track a running minimum and maximum in one pass — the accumulator pattern that separates `fold` from every stateless adaptor.

**Objectives**

- implement `min_max` to fold over the slice with the accumulator `Option<(f64, f64)>`: `None` until the first item, then `(lo, hi)` updated per item — the `None` seed replaces the `f64::MAX/MIN` sentinel
- keep the signature unchanged; no `if` ladder over indices, one `fold`
- read the given check: the empty slice must come back `None`, not `(0.0, 0.0)`

**See**: §7.1 of the example notebook.

**Hint**: seed `None`; in the closure, `match acc { None => Some((*p, *p)), Some((lo, hi)) => Some((lo.min(*p), hi.max(*p))) }`.


In [ ]:
/// (min, max) in one pass; None for an empty slice — the fold carries it all.
///
/// TODO: replace `todo!()` with the Option-seeded fold.
fn min_max(v: &[f64]) -> Option<(f64, f64)> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(min_max(&[4126.5, 4127.0, 4125.5]), Some((4125.5, 4127.0)));
assert_eq!(min_max(&[42.0]), Some((42.0, 42.0))); // one item: min == max
assert_eq!(min_max(&[]), None); // the None seed survives an empty pull
println!("Exercise 7.1 passed");


### Exercise 7.2 — `try_fold`: the fallible accumulator

**Goal**: sum with checked arithmetic, short-circuiting on the first overflow — and then on the first bad parse, the same shape one bridge over.

**Objectives**

- implement `checked_sum` over `&[u64]` with `try_fold` and `checked_add`, returning `Result<u64, &'static str>` (`"overflow"` on the failure)
- implement `checked_parse_sum` over `&[&str]`: parse each row, `?` inside the closure, and sum — the `Err` that stops the fold is whichever parse failed first
- keep the signatures unchanged; the trap row is given as a comment in the check

**See**: §7.2 of the example notebook.

**Hint**: `try_fold(0u64, |acc, x| acc.checked_add(*x).ok_or("overflow"))`; for the parse version the closure body is `Ok(acc + r.parse::<u64>()?)`.


In [ ]:
/// Sum with checked_add: the first overflow becomes Err and stops the pull.
///
/// TODO: replace `todo!()` with the try_fold + checked_add chain.
fn checked_sum(v: &[u64]) -> Result<u64, &'static str> {
    todo!()
}
/// Parse-and-sum in one fold: the first bad row is the error.
///
/// TODO: replace `todo!()` with the try_fold + parse chain.
fn checked_parse_sum(rows: &[&str]) -> Result<u64, std::num::ParseIntError> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(checked_sum(&[1, 2, 3, 4]), Ok(10));
assert_eq!(checked_sum(&[u64::MAX, 1]), Err("overflow")); // the trap row
assert_eq!(checked_parse_sum(&["10", "20"]), Ok(30));
assert!(checked_parse_sum(&["10", "x"]).is_err()); // short-circuited
println!("Exercise 7.2 passed");


### Exercise 8.1 — The pipeline factory: `-> impl Iterator`

**Goal**: write a function that *builds* a chain instead of consuming one — the RPIT return, borrowing its argument, in this project's edition-2024 form.

**Info**: under edition 2024 an RPIT captures all input lifetimes automatically, so the borrowing form compiles with **no** `+ '_` clause; the chain may borrow `v` and the compiler tracks it.

**Objectives**

- implement `in_band` to return the prices of `v` that lie in `[lo, hi]`, as a chain over the borrowed slice — the signature is given and already in the bare edition-2024 form
- keep the signature unchanged; the closure needs `move` — the bounds must live in the iterator, not on the stack frame that ends
- **the stub below ships a placeholder chain** (an RPIT stub cannot be `todo!()` — see §0); make it the real band filter, and the passing asserts replace the placeholder's failure
- observe in the check: the factory is reusable — drive it twice, get independent chains, and `v` survives

**See**: §8.1 of the example notebook.

**Hint**: `v.iter().copied().filter(move |p| *p >= lo && *p <= hi)` — without `move` the closure borrows `lo`/`hi` out of a frame that has returned.


In [ ]:
/// A reusable factory: the prices of v inside [lo, hi], lazily.
///
/// TODO: replace the placeholder chain with the move-closure band filter.
fn in_band(v: &[f64], lo: f64, hi: f64) -> impl Iterator<Item = f64> + '_ {
    v.iter().copied() // placeholder: compiles, wrong answers — swap in the filter
}

// --- check (do not modify) ---
let px = [4120.0, 4126.5, 4130.0, 4126.0];
assert_eq!(in_band(&px, 4125.0, 4128.0).collect::<Vec<f64>>(), vec![4126.5, 4126.0]);
assert_eq!(in_band(&px, 4125.0, 4128.0).count(), 2); // a fresh chain each call
assert_eq!(px.len(), 4); // the argument survived: the chain borrowed it
println!("Exercise 8.1 passed");


### Exercise 8.2 — The boxed escape

**Goal**: return one of *two different chain types* from one function — the E0308 wall from the example notebook, and the `Box<dyn Iterator>` that goes over it.

**Objectives**

- implement `picks` exactly as the example notebook's §8.2 does: `map` on the `true` branch, `filter` on the `false`, both boxed
- confirm from the commented diagnostic why the unboxed version is refused: `Map<..>` and `Filter<..>` are different types, and an RPIT hides exactly one
- keep the signature unchanged

**See**: §8.2 of the example notebook.

**Hint**: `Box::new((0..3).map(|x| x * 10))` in one arm, `Box::new((0..3).filter(|x| x % 2 == 0))` in the other — the box erases the difference.


In [ ]:
/// Two chain types, one return: the box is the escape hatch.
///
/// TODO: replace `todo!()` with the if/else of boxed chains.
fn picks(flag: bool) -> Box<dyn Iterator<Item = i32>> {
    todo!()
}
// The wall the box goes over, from 8.2 of the example notebook:
// error[E0308]: `if` and `else` have incompatible types
//   = note: expected `Map<Range<{integer}>, ...>`
//              found `Filter<Range<{integer}>, ...>`
//   = help: you could change the return type to be a boxed trait object

// --- check (do not modify) ---
assert_eq!(picks(true).collect::<Vec<i32>>(), vec![0, 10, 20]);
assert_eq!(picks(false).collect::<Vec<i32>>(), vec![0, 2]);
println!("Exercise 8.2 passed");


### Exercise 9.1 — Rolling your own adaptor: `Dedup<I>`

**Goal**: write the wrapper that skips repeated items — §2's iterator-with-lookahead, generic over any inner iterator.

**Given**: the `Dedup<I>` struct, its constructor (which primes the `held` lookahead) and its `FusedIterator` impl are complete; only `next` is missing.

**Objectives**

- implement `next` to return the next item that differs from the previously returned one: pull the pending item first (`held.take()`, else `inner.next()`), then skip items equal to it
- keep the struct, constructor and marker impl unchanged; `T: PartialEq` is already on the impls
- read the check: dedup keeps *first occurrences in place* — it does not sort, and empty input ends immediately

**See**: §9.1 of the example notebook.

**Hint**: the body is a loop: take the lookahead (or pull), then `while self.inner.next() == Some(same) {}` — the `?` on the first pull is the early exit for exhaustion.


In [ ]:
/// A std-shaped adaptor: yields each item once, skipping immediate repeats.
struct Dedup<I: Iterator> {
    inner: I,
    held: Option<I::Item>, // one item of lookahead
}
impl<I: Iterator> Dedup<I> {
    fn new(mut inner: I) -> Self {
        let held = inner.next();
        Dedup { inner, held }
    }
}
impl<I: Iterator> Iterator for Dedup<I>
where
    I::Item: PartialEq,
{
    type Item = I::Item;
    /// TODO: replace `todo!()` with the lookahead pull and the skip loop.
    fn next(&mut self) -> Option<Self::Item> {
        todo!()
    }
}
impl<I: Iterator> FusedIterator for Dedup<I> where I::Item: PartialEq {}

// --- check (do not modify) ---
let runs: Vec<i32> = Dedup::new([1, 1, 2, 2, 2, 3, 1].into_iter()).collect();
assert_eq!(runs, vec![1, 2, 3, 1]); // first occurrences in place, not sorted
assert_eq!(Dedup::new(std::iter::empty::<i32>()).next(), None);
let chars: Vec<char> = Dedup::new("aabbca".chars()).collect();
assert_eq!(chars, vec!['a', 'b', 'c', 'a']);
println!("Exercise 9.1 passed");


### Exercise 10.1 — The catalogue, dodged live

**Goal**: use the three stable dodges from the example notebook's §10 — parenthesised ranges, annotated collects, and `rev` on a source that has it.

**Objectives**

- implement `scaled` to map over a range with a *variable* end — the parenthesis is the whole exercise: without it the method call attaches to the bound, E0689
- implement `collected` to build a `Vec<u32>` via `collect` with the turbofish — the E0283 dodge, annotation-in-the-call form
- keep the signatures unchanged; the commented lines below are the diagnostics as captured

**See**: §10.1 of the example notebook.

**Hint**: `(1..end).map(|x| x * 10)` — the parentheses around the range expression; `.collect::<Vec<u32>>()` — the turbofish spelling the target.


In [ ]:
/// Map a range whose end is not a literal: parenthesise or trip E0689.
///
/// TODO: replace `todo!()` with the parenthesised range chain.
fn scaled(end: i32) -> Vec<i32> {
    todo!()
}
/// Collect with the turbofish: the target spelled in the call.
///
/// TODO: replace `todo!()` with the collect::<Vec<u32>> chain.
fn collected() -> Vec<u32> {
    (0..3).map(|x| x * 2).collect::<Vec<u32>>()
}

// --- check (do not modify) ---
assert_eq!(scaled(4), vec![10, 20, 30]);
assert_eq!(scaled(1), vec![]);
assert_eq!(collected(), vec![0, 2, 4]);
// The diagnostics this exercise dodges, as captured on 1.98:
// let bad: Vec<i32> = 1..4i32.map(|x| x * 10).collect();
//   error[E0689]: can't call method `map` on type `i32`
// let n = (0..3).map(|x| x * 2).collect();
//   error[E0283]: type annotations needed
//     = note: the type must implement `FromIterator<i32>`
// let bwd: Vec<i32> = Ticks::new(0, 6, 2).rev().collect();
//   error[E0599]: the trait bound `Ticks: DoubleEndedIterator` is not satisfied
println!("Exercise 10.1 passed");


### Exercise 11.1 — The cost table, re-derived

**Goal**: re-derive the chapter's size facts with `size_of` — the ZST closure, the padded zip, the boxed escape.

**Given**: the helper `size_of_of` from the example notebook's §11.1 is reproduced below; it reads the size of whatever chain you hand it.

**Objectives**

- implement `map_over` to return the `map` chain over a slice iterator — then assert its size equals the bare slice iterator's (the captureless closure is a ZST)
- implement `zipped` to zip a slice iterator with a `u32` range — then assert its size is the padded 40
- implement `boxed_iter` to box the same `map` chain into `Box<dyn Iterator<Item = u32>>` — then assert the box's size is the two-pointer 16
- **these are placeholder-chain stubs** (RPIT cannot be `todo!()` — see §0): the sizes in the check pin the real chains; the placeholder's sizes are wrong and the asserts fail until you swap them in
- keep the signatures unchanged

**See**: §11.1 of the example notebook.

**Hint**: `map_over(v)` is `v.iter().map(|x| x * 2)`; `zipped(v)` is `v.iter().zip(0..v.len() as u32)`; `boxed_iter(v)` wraps the first in `Box::new`.


In [ ]:
fn size_of_of<T>(_: &T) -> usize {
    size_of::<T>()
}
/// The workhorse chain: slice::Iter wrapped in a Map whose closure is a ZST.
///
/// TODO: replace the placeholder with the real map chain.
fn map_over(v: &[u32]) -> impl Iterator<Item = u32> + '_ {
    v.iter().copied() // placeholder: compiles, wrong size — swap in the map
}
/// Zip a slice iterator with a range: two cursors, padded to align.
///
/// TODO: replace the placeholder with the real zip chain.
fn zipped(v: &[u32]) -> impl Iterator<Item = (u32, u32)> + '_ {
    v.iter().map(|x| (*x, *x)) // placeholder: compiles, wrong size and shape
}
/// The boxed escape: the same chain behind a fat pointer.
///
/// TODO: replace the placeholder with Box::new around the map chain.
fn boxed_iter(v: &[u32]) -> Box<dyn Iterator<Item = u32> + '_> {
    Box::new(v.iter().map(|x| x * 2)) // placeholder: right box, wrong inner chain
}

// --- check (do not modify) ---
let data: Vec<u32> = (0..16).collect();
assert_eq!(size_of_of(&map_over(&data)), 16); // 16 = slice::Iter; closure is a ZST
assert_eq!(size_of_of(&zipped(&data)), 40); // 16 + 8, padded
assert_eq!(size_of::<Box<dyn Iterator<Item = u32> + '_>>(), 16); // the fat pointer
assert_eq!(map_over(&data).max(), Some(30));
assert_eq!(zipped(&data).last(), Some((15, 15)));
assert_eq!(boxed_iter(&data).count(), 16);
println!("Exercise 11.1 passed");


## 14. Review: Mixed Problems

Three problems, each assembling several sections' worth of machinery: a full pipeline behind an RPIT factory, a capped fallible accumulation, and the shelf's own infinite iterator — every earlier exercise's move, used once more together.

### Exercise 14.1 — The whole pipeline, one signature

**Goal**: build and consume a complete pipeline in one function — parse, fail fast, transform, bound — with the factory returning the chain for reuse.

**Given**: nothing new; every piece exists from earlier sections (`parse_positions` §3.2, the `Result` collect §5.2, RPIT §8.1, `take` §4.1).

**Objectives**

- implement `clean_totals` to return `Ok` of the first `n` doubled values parsed from `rows`, or the first parse error — `Result` collect and `take` compose in one chain
- implement `in_band_positions` to return the *indices* of the prices inside `[lo, hi]` — `enumerate` feeds the filter, the filter keeps the pair, `map` keeps the index
- keep the signatures unchanged

**See**: §3.2, §4.1 and §5.2 of the example notebook.

**Hint**: after `parse` the chain's items are `Result`s — `take(n)` caps them, `Result::map` doubles each success (`|res| res.map(|x| x * 2)`), and the collect still fails fast. The position filter is `.filter(|(_, p)| **p >= lo && **p <= hi).map(|(i, _)| i)` over `enumerate`.


In [ ]:
/// The first n doubled values, or the first parse error — one chain.
///
/// TODO: replace `todo!()` with the map + take + collect chain.
fn clean_totals(rows: &[&str], n: usize) -> Result<Vec<u32>, std::num::ParseIntError> {
    todo!()
}
/// Indices of the in-band prices — enumerate feeds, filter decides.
///
/// TODO: replace `todo!()` with the enumerate + filter + map chain.
fn in_band_positions(px: &[f64], lo: f64, hi: f64) -> Vec<usize> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(clean_totals(&["10", "20", "30"], 2), Ok(vec![20, 40])); // take before the error
assert!(clean_totals(&["10", "x"], 2).is_err());
assert_eq!(in_band_positions(&[4120.0, 4126.5, 4130.0, 4126.0], 4125.0, 4128.0), vec![1, 3]);
assert_eq!(in_band_positions(&[4120.0], 4125.0, 4128.0), vec![]);
println!("Exercise 14.1 passed");


### Exercise 14.2 — The capped, fallible fold

**Goal**: assemble `try_fold` with a bound and a parse in one fold — the accumulate-stop-cap shape that replaces the `for` loop with three early exits.

**Objectives**

- implement `sum_until_breach` to fold prices into a running total that stops at the first price above `cap`, returning `Err("breach")` — but a price *equal* to the cap is fine, and the running sum is what the cap tests
- implement `fold_parse` to fold parsed rows into a sum that stops at the first unparseable row — the error is the parse error itself, not a string
- keep the signatures unchanged

**See**: §7.1, §7.2 and §4.2 of the example notebook.

**Hint**: `try_fold(0u64, |acc, p| if acc + *p > cap { Err("breach") } else { Ok(acc + *p) })` — the accumulator is the state the predicate reads; for the parse fold, `Ok(acc + r.parse::<u64>()?)`.


In [ ]:
/// Running total under a cap; the first breach price aborts with Err.
///
/// TODO: replace `todo!()` with the try_fold + cap check.
fn sum_until_breach(px: &[u64], cap: u64) -> Result<u64, &'static str> {
    todo!()
}
/// Sum of parsed rows; the first unparseable row is the error.
///
/// TODO: replace `todo!()` with the try_fold + parse chain.
fn fold_parse(rows: &[&str]) -> Result<u64, std::num::ParseIntError> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(sum_until_breach(&[10, 20, 30], 60), Ok(60)); // lands exactly on the cap
assert_eq!(sum_until_breach(&[10, 20, 40], 60), Err("breach")); // 30 + 40 exceeds
assert_eq!(sum_until_breach(&[], 60), Ok(0)); // empty: the seed survives
assert_eq!(fold_parse(&["10", "20"]), Ok(30));
assert!(fold_parse(&["10", "x"]).is_err());
println!("Exercise 14.2 passed");


### Exercise 14.3 — The shelf's own infinite iterator

**Goal**: implement `Fib` — the canonical infinite source — then consume it three ways, bounded every time: the chapter's laziness, short-circuiting and `take`-bounding in one cell.

**Given**: the `Fib` struct and its constructor are complete; only `next` is missing. The empty `FusedIterator` impl is given — your `next` will honour it.

**Objectives**

- implement `next` to yield the Fibonacci sequence forever, advancing the `(a, b)` pair each pull
- keep the struct, constructor and marker impl unchanged
- the check consumes `Fib::new()` three ways — all bounded; an unbounded collect anywhere hangs, which is the §4 lesson in negative

**See**: §2.1 and §4.1 of the example notebook.

**Hint**: the cursor is a pair: `let out = self.a; self.a = self.b; self.b = out + self.b; Some(out)` — pull, shift, grow.


In [ ]:
/// The canonical infinite source: next() never returns None.
struct Fib {
    a: u64,
    b: u64,
}
impl Fib {
    fn new() -> Self {
        Fib { a: 0, b: 1 }
    }
}
impl Iterator for Fib {
    type Item = u64;
    /// TODO: replace `todo!()` with the pull-shift-grow body.
    fn next(&mut self) -> Option<u64> {
        todo!()
    }
}
impl FusedIterator for Fib {}

// --- check (do not modify) ---
let first: Vec<u64> = Fib::new().take(8).collect();
assert_eq!(first, vec![0, 1, 1, 2, 3, 5, 8, 13]);
assert_eq!(Fib::new().position(|x| x > 10), Some(7)); // 13, at index 7
let evens: Vec<u64> = Fib::new().filter(|x| x % 2 == 0).take(3).collect();
assert_eq!(evens, vec![0, 2, 8]); // laziness: the filter saw the whole prefix
println!("Exercise 14.3 passed");


### What Comes Next

Chapter 12's iterators are now yours to write without looking anything up. The track continues:

| Chapter | Title | What it settles |
|---|---|---|
| **13** | **Modules, Crates and Testing** | `pub` finally bites, and every `#[test]` you write is a chain like these |
| 14 | Smart Pointers and Interior Mutability | `Rc`/`RefCell` — the sharing §4.1's counter borrowed, properly introduced |
| 15 | Data Types and Their Libraries | `polars` — iterators over columns, at library scale |

**Carry forward.** The seven moves this notebook drilled: driving `next` by hand, implementing an iterator from scratch (with the exact-hint upgrade), the adaptor/consumer split, bounding infinite sources with `take`, choosing and spelling `collect`'s target, the stateful `fold` and fallible `try_fold`, and returning pipelines — RPIT for one chain, the box for two.
